# Training Deformable DETR With SAM Backbone for Cell Instance Detection

## Preparations
### Import required libraries

In [ ]:
import os
import sys
import time
import pickle
import json
from typing import Tuple, Union, List, Dict, Final

import numpy as np
from PIL import Image
import cv2
import torch
import pycocotools
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask_util
import albumentations as A
from transformers import AutoImageProcessor
# sys.path.append('references/detection')
# from references.detection.coco_utils  import CocoDetection
from torchvision.datasets import CocoDetection

## Pre-processing configurations
The following configurations are used for pre-processing the images during the training. 

In [ ]:
# probability of adding random blur and salt-and-pepper or additive gaussian noise to the training images
# see the image Transforms section below
P_NOISE = 0.25

# the lower and upper bounds for random scaling the images (and annotated masks) for training augmentation
MIN_RANDOM_SCALE = 0.7
MAX_RANDOM_SCALE = 1.0

# model input image size (should be square)
MODEL_INPUT_SIZE: Final[int] = 1024

MIN_AREA: Final[int] = 25

### Transforms
We are using `Albumentations` package for all image and annotation augmentations. We use the Hugging face AutoImageProcessor class for preprocessing the images.

In [ ]:
def get_transform(train: bool = True) -> A.core.composition.Compose:
    # no resizing is needed here as the training images are already cropped with the model input size, 
    # and also the Hugging face processor will handle any needed resizing
    if train:
        # random noise addition and random scaling/rotation
        trsfms = [
            A.RandomScale(scale_limit=(MIN_RANDOM_SCALE - 1.0, MAX_RANDOM_SCALE - 1.0), p=1.0),
            A.PadIfNeeded(min_height = MODEL_INPUT_SIZE, min_width = MODEL_INPUT_SIZE, position = 'random'),
            A.RandomCrop(height = MODEL_INPUT_SIZE, width=MODEL_INPUT_SIZE, pad_if_needed=False, p=1.0),
            A.RandomRotate90(),
            A.Perspective(p=P_NOISE),
            A.RandomBrightnessContrast(p=P_NOISE),
            A.HueSaturationValue(p=P_NOISE),
            A.AdditiveNoise(p=P_NOISE, spatial_mode='per_pixel')
        ]        
    else:
        # test set 
        trsfms = [A.NoOp()]
    # bbox format is defined as COCO xywh
    return A.Compose(trsfms, bbox_params=A.BboxParams(format="coco", label_fields=["category"], clip=True, min_area=MIN_AREA))

# hugging face 
# we need to resize the input images (not needed as they are already in the correct MODEL_INPUT_SIZE x MODEL_INPUT_SIZE input size, 
# and normalize them, we use the already implemented Hugging Face preprocessor for this conversion
# index 0 will be used 

# from transformers import SamProcessor
# hg_preprocessor = SamProcessor.from_pretrained("facebook/sam-vit-base")

from transformers import DeformableDetrImageProcessor
hg_preprocessor = DeformableDetrImageProcessor.from_pretrained("SenseTime/deformable-detr")
# modify the resizing part as we will replace the backbone with that of SAM
hg_preprocessor.size = {'longest_edge': MODEL_INPUT_SIZE, 'shortest_edge': MODEL_INPUT_SIZE}

## Data Model
The dataset class as well as the training and evaluation scripts are adopted from the RT-DETR fine-tuning tuturial Notebook here:  https://github.com/NielsRogge/Transformers-Tutorials/blob/master/RT-DETR/Fine_tune_RT_DETR_on_a_custom_dataset.ipynb. 

NOTE: The labels (class IDs) for both YOLO and RT-DETR models starts with 0.

### Dataset for already pre-processed annotated data
This dataset is built by passing the location of the pre-processed images and corresponding json annotation file in COCO format. The code below assumes:
1. The train/test data is already cropped and the annotation files are already created in the required Mask R-CNN or YOLO format.
2. `convert_to_coco_api` function has been run on them to generate the json annotation file in COCO formet.

The dataset class takes care of image augmentation and any preprocessing that is required for the model. 

In [ ]:
class CellMaskDataset(torch.utils.data.Dataset):
    def __init__(self, 
                 dataset_coco: CocoDetection, 
                 processor: AutoImageProcessor, 
                 instance_segmentation: bool = False,
                 transforms: A.core.composition.Compose = None):
        self.dataset_coco = dataset_coco
        self.processor = processor
        self.instance_segmentation = instance_segmentation
        self.transforms = transforms

   
    def __len__(self):
        return len(self.dataset_coco)

    def __getitem__(self, idx):
        image, annotations = self.dataset_coco[idx]
        
        # Convert image to RGB numpy array
        image_array: np.ndarray = np.array(image.convert("RGB"))
        
        if isinstance(annotations, dict):
            # references.detection.coco_utils.CocoDetection class, annotations is a dictionary with keys image_id, annotations
            # annotations['annotations'] is a list of annotation dictionaries
            image_id: int = annotations['image_id']
            boxes: np.ndarray = np.array([record['bbox'] for record in annotations['annotations']])
            labels: List[int] = [record['category_id'] for record in annotations['annotations']]
            masks: List[np.ndarray] = [coco_mask_util.decode(record['segmentation']) for record in annotations['annotations']]
        else:
            # torchvision.datasets.CocoDetection class, annotations is a list of dictionaries for each annotated object
            if len(annotations) > 0:
                image_id: int = annotations[0]['image_id']
            else:
                image_id: int = idx + 1
            boxes: np.ndarray = np.array([record['bbox'] for record in annotations])
            labels: List[int] = [record['category_id'] for record in annotations]
            masks: List[np.ndarray] = [coco_mask_util.decode(record['segmentation']) for record in annotations]
        
        # apply augmentations, the second condition should not happen (all preprocessed images should at least have one object)
        if self.transforms and len(boxes) > 0:
            if self.instance_segmentation:
                # TODO: check to make sure the below mask transform would work
                transformed = self.transforms(image=image_array, bboxes=boxes, masks=masks, category=labels)
                img = transformed["image"]
                boxes = transformed["bboxes"]
                masks = transformed["masks"]
                labels = transformed["category"]
            else:
                transformed = self.transforms(image=image_array, bboxes=boxes, category=labels)
                img = transformed["image"]
                boxes = transformed["bboxes"]
                labels = transformed["category"]

        # reformat annotations
        annotations: List[dict] = []
        for i, bbox in enumerate(boxes):
            record = {
                "image_id": image_id,
                "category_id": int(labels[i]),
                "bbox": np.array([int(v) for v in bbox]),
                "iscrowd": 0,
                "area": bbox[2] * bbox[3],
            }
            if self.instance_segmentation:
                record["segmentation"] = coco_mask_util.encode(np.asarray(masks[i], order="F"))
                record["segmentation"]['counts'] = record["segmentation"]['counts'].decode('utf8')
            annotations.append(record)

        formatted_annotation: dict = {"image_id": image_id, "annotations": annotations,}
        if self.processor is not None:
            results = self.processor(images=img, annotations=formatted_annotation, return_tensors="pt")

            # image processor expands batch dimension, lets squeeze it
            results = {k: v.squeeze() if isinstance(v, torch.Tensor) else v[0] for k, v in results.items()}
            return results
         
        return img, formatted_annotation

#### Dataset prepration
----------------
The code assumes the train and test json annotation files are named `train_annotations.json` and `test_annotations.json`, respectively, and are placed under the root dataset folder, and the train and test images are stored under `images/train` and `images/test` folders under the root dataset folder.

Make sure the LABEL_MAP below is consistent with the way the annotations are parsed (the mapping between the annotation class names in the raw annotation format and the mapped class IDs), and the mapping between the model class IDs and names. 

In [ ]:
# make sure these folders are generated in advance
# 
BASE_PATH = '/home/cellareye/Cellanome/dl-mehdi/Mask RCNN/data/20240625_mc38_10x_caged_cropped_for_SAM_5_class'
TRAIN_IMAGE_FOLDER = os.path.join(BASE_PATH, 'images', 'train')
TEST_IMAGE_FOLDER =os.path.join(BASE_PATH, 'images', 'test')


# mapping between the class IDs and class names for the annotated data 
# labels start from 0
LABEL_MAP = {0: 'cell', 1: 'bead', 2: 'cage', 3: 'cell-adhered', 4: 'some'}
REVERESE_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

In [ ]:
# read the prepared json annotations in COCO format as COCO dataset objects
train_dataset_coco = CocoDetection(root=TRAIN_IMAGE_FOLDER, annFile=os.path.join(BASE_PATH, 'train_annotations.json'), transforms=None)
test_dataset_coco = CocoDetection(root=TEST_IMAGE_FOLDER, annFile=os.path.join(BASE_PATH, 'test_annotations.json'), transforms=None)

train_dataset = CellMaskDataset(
    dataset_coco=train_dataset_coco, 
    processor=hg_preprocessor,
    instance_segmentation=False, # no need to set to True for training a DETR model as it adds to runtime computation! set True for testing
    transforms=get_transform(True)
)

test_dataset = CellMaskDataset(
    dataset_coco=test_dataset_coco, 
    processor=hg_preprocessor,
    instance_segmentation=False, # no need to set to True for training a DETR model as it adds to runtime computation
    transforms=get_transform(False)
)

## Model Definition

In [ ]:
from transformers import DeformableDetrConfig, DeformableDetrForObjectDetection

def get_deformable_detr_model(
    id2label: Dict[int, str], 
):
    # pre_trained_model_checkpoint: str = "SenseTime/deformable-detr"
    label2id: Dict[str, int] =  {v: k for k, v in id2label.items()}
    # model = DeformableDetrForObjectDetection.from_pretrained(
    #     pre_trained_model_checkpoint,
        # id2label=id2label,
        # label2id=label2id,
        # ignore_mismatched_sizes=True,
    # )
    # replace the backbone with SAM
    # model_config = model.config
    # model_config.use_timm_backbone = True
    # model_config.backbone = 'samvit_base_patch16.sa1b'
    # model_config.use_pretrained_backbone = True
    # model_config.id2label = id2label
    # model_config.label2id = label2id
   

    model_config = DeformableDetrConfig(use_timm_backbone = True, 
                                        backbone = 'samvit_base_patch16.sa1b', 
                                        use_pretrained_backbone = True,
                                        id2label=id2label, 
                                        label2id = label2id)
    model = DeformableDetrForObjectDetection(model_config)
    # this is for freezing the backbone
    for param in model.model.backbone.parameters():
        param.requires_grad_(False)

    return model

## Training
### Training parameters

In [ ]:
TRAIN_BATCH_SIZE = 4
LEARNING_RATE = 5e-5
NUM_EPOCHS = 25
MODEL_PATH = 'checkpoints'

### Data loaders

In [ ]:
def collate_fn(batch):
    data = {}
    data["pixel_values"] = torch.stack([x["pixel_values"] for x in batch])
    data["pixel_mask"] = torch.stack([x["pixel_mask"] for x in batch])
    data["labels"] = [x["labels"] for x in batch]
    return data

# define training and validation data loaders
train_data_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size = TRAIN_BATCH_SIZE, shuffle = True, num_workers = 2,
    collate_fn = collate_fn)

test_data_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size = 1, shuffle = False, num_workers = 2,
    collate_fn = collate_fn)

print('Training data includes %d annotated images.' %len(train_dataset))
print('Test data includes %d annotated images.' %len(test_dataset))

### Visually checking some data

In [ ]:
import torchvision
from transformers.image_transforms import center_to_corners_format

def convert_bbox_yolo_to_pascal(boxes, image_size):
    """
    Convert bounding boxes from YOLO format (x_center, y_center, width, height) in range [0, 1]
    to Pascal VOC format (x_min, y_min, x_max, y_max) in absolute coordinates.

    Args:
        boxes (torch.Tensor): Bounding boxes in YOLO format
        image_size (Tuple[int, int]): Image size in format (height, width)

    Returns:
        torch.Tensor: Bounding boxes in Pascal VOC format (x_min, y_min, x_max, y_max)
    """
    # convert center to corners format
    boxes = center_to_corners_format(boxes)

    # convert to absolute coordinates
    height, width = image_size
    boxes = boxes * torch.tensor([[width, height, width, height]])
    return boxes

COLORS = [(0, 0, 0), (0, 0, 255), (255, 0, 0), (0, 255, 0), (255, 255, 0), (255, 0, 255)]

def show_sample(idx, dataset):
    sample = dataset[idx]
    if isinstance(sample, tuple):
        image, annotations = sample
        boxes = np.array([t['bbox'] for t in annotations['annotations']])
        if len(boxes) > 0:
            boxes[:, 2] += boxes[:, 0]
            boxes[:, 3] += boxes[:, 1]
        labels = np.array([record['category_id'] for record in annotations['annotations']])
        if len(annotations['annotations']) and 'segmentation' in  annotations['annotations'][0]:
            masks = np.array([coco_mask_util.decode(record['segmentation']) for record in annotations['annotations']])
        else:
            masks = None
    else:
        set_mean = torch.tensor(dataset.processor.image_mean)
        set_var = torch.tensor(dataset.processor.image_std)
        image = (set_mean + sample['pixel_values'].permute(1, 2, 0) * set_var).mul(255).byte().numpy().copy()
        boxes = convert_bbox_yolo_to_pascal(sample['labels']['boxes'], 
                                            (sample['labels']['size'][0].item(), sample['labels']['size'][0].item())).numpy().astype(int)
        labels = sample['labels']['class_labels'].numpy().tolist()
        masks = None
                   
                
    for i in range(len(labels)):
        # the bounding box
        (xtl, ytl, xbr, ybr) = boxes[i]
        # use green color for masks
        color = COLORS[(labels[i] + 1) % len(COLORS)] # add 1 to be consistent with Mask R-CNN colors/labels
        if masks is not None:
            color_mask = color * np.repeat(np.expand_dims(masks[i][ytl:ybr, xtl:xbr], axis=2), 3, axis=2)
            blended = 0.4 * color_mask
            blended[color_mask == 0] = image[ytl:ybr, xtl:xbr][color_mask == 0]
            blended[color_mask > 0] += 0.6 * image[ytl:ybr, xtl:xbr][color_mask > 0]

            # store the blended ROI in the original image
            image[ytl:ybr, xtl:xbr] = blended.astype(np.uint8)
        
        if labels[i] in LABEL_MAP:
            text = LABEL_MAP[labels[i]]
        else:
            print('Incorrect ID was found %s' %labels[i])
            text = 'Unknown'
        
        # add label
        cv2.putText(image, text, (xtl, ytl + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        # add the bounding box with yellow color
        color = (255, 255, 0)
        cv2.rectangle(image, (xtl, ytl), (xbr, ybr), color, 1)
        
    print(f"Image size (W, H): {image.shape[1]}, {image.shape[0]}")
    # convert to PIL image to display
    return Image.fromarray(image)

In [ ]:
display(show_sample(343, train_dataset))

### Model selection
### From a pretrained model on COCO (scratch)

In [ ]:
model = get_deformable_detr_model(
    id2label=LABEL_MAP
)

# train on the GPU or on the CPU, if a GPU is not available
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

print('Device available:' , device)

# move model to the right device
model.train()
model.to(device)

### From an already trained model on our dataset

In [ ]:
pre_trained_model_checkpoint: str = "checkpoints/checkpoint-112300"
model = DeformableDetrForObjectDetection.from_pretrained(
        pre_trained_model_checkpoint,
        id2label=LABEL_MAP,
        label2id={v: k for k, v in LABEL_MAP.items()},
        ignore_mismatched_sizes=False,
    )
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval()

### Evalutation function
A function to compute COCO mAP and mAR.

In [ ]:
from pycocotools.cocoeval import COCOeval

def to_cpu_device(tensor):
    """
    A function to move a CUDA torch input to CPU memory.
    Args:
        tensor (torch tensor).
    Returns:
        Moved to CPU.
    """
    return tensor.detach().cpu() if tensor.requires_grad else tensor.cpu()

def convert_to_xywh(boxes):
    xmin, ymin, xmax, ymax = boxes.unbind(1)
    return torch.stack((xmin, ymin, xmax - xmin, ymax - ymin), dim=1)


def convert_preds_to_coco(predictions):
    coco_results = []
    for original_id, prediction in predictions.items():
        if len(prediction) == 0:
            continue
        
        boxes = prediction["boxes"]
        boxes = convert_to_xywh(boxes).tolist()
        
        scores = prediction["scores"].tolist()
        labels = prediction["labels"].tolist()
         
        coco_results.extend(
            [
                {
                    "image_id": original_id,
                    "category_id": labels[k],
                    "bbox": boxes[k],
                    "score": scores[k],
                }
                for k in range(len(scores))
            ]
        )
    return coco_results

from dataclasses import dataclass

@dataclass
class ModelOutput:
    logits: torch.Tensor
    pred_boxes: torch.Tensor


class MAPEvaluator:

    def __init__(self, data_loader, image_processor, threshold=0.4, max_dets=100):
        self.image_processor = image_processor
        self.threshold = threshold
        self.test_dataset_coco = data_loader.dataset.dataset_coco
        self.all_predictions = []
        self.all_image_ids = []
        self.max_dets = max_dets

    def collect_image_sizes(self, batch):
        """Collect image sizes across a batch of the dataset as a list of batch_size size (list of 2 elements, height and width)."""
        # we should use size and not 'org_size' this is because we directly use ground truth from the COCO test dataset, which already
        # have the images resized to size
        batch_image_sizes = [to_cpu_device(x["size"]).numpy().tolist() for x in batch]
        return batch_image_sizes

    def collect_predictions(self, batch_predictions, batch_image_sizes):
        post_processed_predictions = []
        batch_logits, batch_boxes = batch_predictions[1], batch_predictions[2]
        output = ModelOutput(logits=batch_logits, pred_boxes=batch_boxes)
        # post_processed_output is a list of batch_size dictionaty elements, each dictionary containing the detections
        # for the images in the batch with keys as 'boxes', 'labels' and 'scores', and values as
        # - 'boxes': a (num_detection, 4) torch.float32 tensor of bounding boxes in (xtl, ytl, xbr, ybr) format
        # - 'labels': a (num_detection, 1) torch.int64 tensor of class IDs
        # - 'scores': a (num_detection, 1) torch.float32 tensor of detection confidences
        post_processed_output = self.image_processor.post_process_object_detection(
            output, threshold=self.threshold, target_sizes=batch_image_sizes
        )        
        # move the detections to CPU
        post_processed_output = [{k: to_cpu_device(v) for k, v in outputs.items()} for outputs in post_processed_output]
        post_processed_predictions.extend(post_processed_output)
        return post_processed_predictions
    
    # metrics should be a dictionary with the following keys: 
    # 'map', 'map_50', 'map_75', 'map_small', 'map_medium', 'map_large', 'mar_1', 'mar_10', 'mar_100', 'mar_small', 'mar_medium', 'mar_large', 
    # 'map_cell', 'mar_100_cell', 'map_bead', 'mar_100_bead', 'map_soma', 'mar_100_soma'
    @torch.no_grad()
    def __call__(self, evaluation_results, compute_result):

        metrics = {
                'map': -1.0, 'map_50': -1.0, 'map_75': -1.0, 
                'map_small': -1.0, 'map_medium': -1.0, 'map_large': -1.0, 
                'mar_1': -1.0, 'mar_10': -1.0, 'mar_' + str(self.max_dets): -1.0, 
                'mar_small': -1.0, 'mar_medium': -1.0, 'mar_large': -1.0
            }

        batch_predictions, batch_targets = evaluation_results.predictions, evaluation_results.label_ids 
        
        batch_image_sizes = self.collect_image_sizes(batch_targets)
        post_processed_batch_predictions = self.collect_predictions(batch_predictions, batch_image_sizes)
        results = {int(target["image_id"].item()): output for target, output in zip(batch_targets, post_processed_batch_predictions)}    
        results = convert_preds_to_coco(results)
        self.all_predictions.extend(results)
        self.all_image_ids += [int(target["image_id"].item()) for target in batch_targets]
        
        if compute_result and len(self.all_predictions) > 0:
            n_threads = torch.get_num_threads()
            # FIXME remove this and make paste_masks_in_image run on the GPU
            torch.set_num_threads(1)
            cpu_device = torch.device("cpu")
    
            coco_gt = self.test_dataset_coco.coco  
            coco_dt = coco_gt.loadRes(self.all_predictions)  # init predictions api
    
            evaluator_time = time.time()
    
            # bounding box evaluation
            coco_evaluator_bbox = COCOeval(coco_gt, coco_dt, "bbox")
            coco_evaluator_bbox.params.maxDets = [1, 10, self.max_dets]
            coco_evaluator_bbox.params.imgIds = self.all_image_ids
            coco_evaluator_bbox.evaluate()
            coco_evaluator_bbox.accumulate()
            coco_evaluator_bbox.summarize()
            evaluator_time = time.time() - evaluator_time
    
            print("evaluator_time:", evaluator_time)

            torch.set_num_threads(n_threads)
            for i, key in enumerate(metrics.keys()):
                metrics[key] = coco_evaluator_bbox.stats[i]
            
            metrics = {k: round(v.item(), 4) for k, v in metrics.items()}

            # clear up the history
            self.all_predictions = []
            self.all_image_ids = []
        
        if compute_result:
            self.all_image_ids = []
            
        return metrics
# we should use a very small threshold here to allow MAPEvaluator to use all the predictions with their scores
eval_compute_metrics_fn = MAPEvaluator(data_loader=test_data_loader, image_processor=hg_preprocessor, threshold=0.01, max_dets=100)

### Run Training using Hugging face training script

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=MODEL_PATH,
    num_train_epochs=NUM_EPOCHS,
    max_grad_norm=0.1,
    learning_rate=LEARNING_RATE,
    warmup_steps=300,
    lr_scheduler_type = "reduce_lr_on_plateau",
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    torch_empty_cache_steps=int(len(train_dataset) / (5 * TRAIN_BATCH_SIZE)), 
    batch_eval_metrics=True,
    dataloader_num_workers=4,
    metric_for_best_model="eval_map",
    greater_is_better=True,
    load_best_model_at_end=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    remove_unused_columns=False,
    eval_do_concat_batches=False,
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    # tokenizer=hg_preprocessor,
    processing_class=hg_preprocessor,
    data_collator=collate_fn,
    compute_metrics=eval_compute_metrics_fn,
)

trainer.train()

In [ ]:
# load the model, the latest saved checkpoint will be loaded
pre_trained_model_checkpoint: str = "checkpoints/checkpoint-112300"
model = DeformableDetrForObjectDetection.from_pretrained(
        pre_trained_model_checkpoint,
        id2label=LABEL_MAP,
        label2id={v: k for k, v in LABEL_MAP.items()},
        ignore_mismatched_sizes=False,
    )

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)


DETECTION_REMAP = None
# DETECTION_REMAP = {"cell-adhered": "cell", "soma": "cell"}


RESIZE: Final[Dict[Tuple[int, int, str], Tuple[int, int]]] = {
    (2000, 1600, "10x"): (1280, 1024),
    (4512, 4512, "10x"): (3328, 3328),
    (4512, 4512, "4x"): (3328, 3328),
}
# A dictionary with keys as the input (original) image size (width, height, magnification)
# tuple and values as the list of coordinates (xtl, ytl, xbr, ybr) of sub-images/crops
# to run YOLOv5 on each
# note that the crop coordinates are with respect to resized image dimensions specified above
CROP_CORNERS: Final[Dict[Tuple[int, int, str], List[List[int]]]] = {
    (2000, 1600, "10x"): [
        [0, 0, 1024, 1024], 
        [256, 0, 1280, 1024]
    ],
    (4512, 4512, "10x"): [
        [0, 0, 1024, 1024],
        [0, 768, 1024, 1792],
        [0, 1536, 1024, 2560],
        [0, 2304, 1024, 3328],
        [768, 0, 1792, 1024],
        [768, 768, 1792, 1792],
        [768, 1536, 1792, 2560],
        [768, 2304, 1792, 3328],
        [1536, 0, 2560, 1024],
        [1536, 768, 2560, 1792],
        [1536, 1536, 2560, 2560],
        [1536, 2304, 2560, 3328],
        [2304, 0, 3328, 1024],
        [2304, 768, 3328, 1792],
        [2304, 1536, 3328, 2560],
        [2304, 2304, 3328, 3328]
    ],
    (4512, 4512, "4x"): [
        [0, 0, 1024, 1024],
        [0, 768, 1024, 1792],
        [0, 1536, 1024, 2560],
        [0, 2304, 1024, 3328],
        [768, 0, 1792, 1024],
        [768, 768, 1792, 1792],
        [768, 1536, 1792, 2560],
        [768, 2304, 1792, 3328],
        [1536, 0, 2560, 1024],
        [1536, 768, 2560, 1792],
        [1536, 1536, 2560, 2560],
        [1536, 2304, 2560, 3328],
        [2304, 0, 3328, 1024],
        [2304, 768, 3328, 1792],
        [2304, 1536, 3328, 2560],
        [2304, 2304, 3328, 3328]
    ]
}

model_param_dict = {}
model_param_dict['model_state_dict'] = model.state_dict()
model_param_dict['label_map'] = LABEL_MAP
model_param_dict['resize_dict'] = RESIZE
model_param_dict['crop_corners_dict'] = CROP_CORNERS
if DETECTION_REMAP is not None:
    model_param_dict['detected_class_names_remap'] = DETECTION_REMAP

torch.save(model_param_dict, os.path.join(MODEL_PATH, 'final.pt'))

In [ ]:
from pprint import pprint

metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="eval")
pprint(metrics)

### Run Inference
To run inference on a previously trained model, run the cells above up to "Cell size analysis".

In [ ]:
from torchvision.transforms import functional as F
from transformers import RTDetrImageProcessor

def to_numpy(tensor):
    """
    A function to convert a torch input to numpy array.
    Args:
        tensor (torch tensor).
    Returns:
        Converted to numpy array.
    """
    return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()

def show_predictions(image_pil, predictions, color_depth=12):
    # convert to a numpy array
    image = np.array(image_pil)
    # scale
    image = (255 * image.astype(float) / (2**color_depth - 1)).astype(np.uint8)
    # convert to 3-channels
    image = np.repeat(np.expand_dims(image, axis=2), 3, axis=2)

    boxes = predictions['boxes']
    labels = predictions['labels']

    for i in range(len(boxes)):
        # the bounding box
        (xtl, ytl, xbr, ybr) = boxes[i].astype(int)
        # use green color for masks
        color = COLORS[(labels[i] + 1) % len(COLORS)]
        
        if labels[i] in LABEL_MAP:
            text = LABEL_MAP[labels[i]]
        else:
            print('Incorrect ID was found %s' %labels[i])
            text = 'Unknown'
        
        # add label
        cv2.putText(image, text, (xtl, ytl + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        cv2.rectangle(image, (xtl, ytl), (xbr, ybr), color, 1)
        
    # convert to PIL image to display
    return Image.fromarray(image)

# we return the results in the same format as Mask R-CNN and YOLO to be able to reuse the codes written for that model
def predict_batch(model, input_images_list, device, detection_threshold=0.4):
    
    model.eval()
    model.to(device)

    # convert to 3-channel images if needed, and store the original image dimensions for 
    # post processing
    images_list: List[np.array] = []
    org_img_dims: List[Tuple[int, int]] = []
    
    for img in input_images_list:
        img_shape: tuple = img.shape
        if len(img_shape) < 3:
            images_list.append(cv2.cvtColor(img, cv2.COLOR_GRAY2RGB))
        else:
            images_list.append(img)
        org_img_dims.append(img_shape[:2])
    
    hg_preprocessor = DeformableDetrImageProcessor(
        do_convert_annotations=True,
        do_resize=True,
        size={"width": MODEL_INPUT_SIZE, "height": MODEL_INPUT_SIZE},
        reduce_labels=False,
        do_rescale=True, 
        do_normalize=True
    )
       
    processed_imgs_dict = hg_preprocessor(images_list, return_tensors="pt")
    with torch.no_grad():
        outputs = model(pixel_values=processed_imgs_dict["pixel_values"].to(device))
        
        processed_outputs = hg_preprocessor.post_process_object_detection(
            outputs, threshold=detection_threshold, target_sizes=org_img_dims
        )

    # processed_outputs is a list of len(input_images_list) dictionary elements, each dictionary containing the detections
    # for the input image in the input list with keys as 'boxes', 'labels' and 'scores', and values as
    # - 'boxes': a (num_detection, 4) torch.float32 tensor of bounding boxes in (xtl, ytl, xbr, ybr) format
    # - 'labels': a (num_detection, 1) torch.int64 tensor of class IDs
    # - 'scores': a (num_detection, 1) torch.float32 tensor of detection confidences
        
    if len(processed_outputs) == 0:
        # this should not happen and is not expected, return as if the model has not detected anything (for the whole list of images)
        return [
            {'boxes': [],
             'labels': [],
             'scores': [],
            }
        ] * len(images_list)

    # move to CPU and convert to numpy arrays before returning
    return [{k: to_numpy(v) for k, v in result.items()} for result in processed_outputs]

In [ ]:
pre_trained_model_checkpoint: str = "checkpoints/checkpoint-112300"
model = DeformableDetrForObjectDetection.from_pretrained(
        pre_trained_model_checkpoint,
        id2label=LABEL_MAP,
        label2id={v: k for k, v in LABEL_MAP.items()},
        ignore_mismatched_sizes=False,
    )
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval()

In [ ]:
# idx = 145123
idx = 13
# img_path = os.path.join(test_dataset.images_path, test_dataset.imgs[idx])
img_id = test_dataset.dataset_coco.coco.getImgIds()[idx]
img_path = os.path.join(test_dataset.dataset_coco.root, test_dataset.dataset_coco.coco.imgs[img_id]['file_name'])
img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)

In [ ]:
out = predict_batch(model, [img], device, 0.45)[0]

In [ ]:
display(show_predictions(img, out, 8))

In [ ]:
display(show_sample(idx, test_dataset))

### Measuring the run-time

In [ ]:
import time
start = time.time()
for i in range(10):
    out = predict_batch(model, [img], device)[0]
print('Running RT-DETR took {} ms'.format((time.time() - start) * 100))